# Unified EEG Pipeline (LOOCV + Cache + 3-Model Voting)

这个 notebook 把你之前分散的逻辑统一到一个可运行流程里：
- 统一特征提取
- 统一特征选择
- 三模型投票
- 默认 3 病人 LOOCV
- 结果和临床可视化
- 特征缓存，避免每次重复读取 EDF


## 1) Imports 与全局配置
> 这里集中放依赖、数据结构、默认参数。

In [ ]:
import hashlib
import json
import os
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import mne
import numpy as np
from scipy.integrate import simpson
from scipy.signal import butter, filtfilt, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


@dataclass
class EpochMetadata:
    file_name: str
    start_s: float
    end_s: float


DEFAULT_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
}


## 2) 数据解析与通道对齐
> 解析 summary 文件，获取所有病人共同通道，保证跨病人训练/测试维度一致。

In [ ]:
def parse_summary_to_dict(summary_path: str) -> Dict[str, List[Tuple[float, float]]]:
    with open(summary_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f.readlines()]

    seizure_dict: Dict[str, List[Tuple[float, float]]] = {}
    current_file = None
    starts: List[float] = []
    ends: List[float] = []

    def flush() -> None:
        nonlocal current_file, starts, ends
        if current_file is not None:
            seizure_dict[current_file] = list(zip(starts, ends))
        starts, ends = [], []

    for line in lines:
        m_file = re.match(r"File Name:\s*(.*)", line)
        if m_file:
            flush()
            current_file = m_file.group(1).strip()
            continue

        m_s = re.match(r"Seizure Start Time:\s*(\d+)\s*seconds", line)
        m_e = re.match(r"Seizure End Time:\s*(\d+)\s*seconds", line)
        if m_s and current_file is not None:
            starts.append(float(m_s.group(1)))
        if m_e and current_file is not None:
            ends.append(float(m_e.group(1)))

    flush()
    return seizure_dict


def list_edf_files(folder: str) -> List[str]:
    edf_paths = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".edf")])
    if not edf_paths:
        raise FileNotFoundError(f"No EDF files found in: {folder}")
    return edf_paths


def normalize_channel_names(raw: mne.io.BaseRaw) -> None:
    if "T8-P8-0" in raw.ch_names and "T8-P8" not in raw.ch_names:
        mne.rename_channels(raw.info, {"T8-P8-0": "T8-P8"})


def find_common_channels(patient_dirs: Sequence[str]) -> List[str]:
    common: Iterable[str] | None = None
    for patient_dir in patient_dirs:
        for path in list_edf_files(patient_dir):
            raw = mne.io.read_raw_edf(path, preload=False, verbose=False)
            normalize_channel_names(raw)
            channels = set(raw.ch_names)
            raw.close()
            if common is None:
                common = channels
            else:
                common = set(common).intersection(channels)
    if not common:
        raise ValueError("No common channels across selected patients.")
    return sorted(common)


## 3) 信号分段 + 特征工程
> 输出每个 epoch 的空间特征，再做 temporal stacking。

In [ ]:
def bandpass_filter_multich(data: np.ndarray, fs: float, l_freq: float, h_freq: float, order: int = 4) -> np.ndarray:
    nyq = 0.5 * fs
    low = l_freq / nyq
    high = h_freq / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, data, axis=-1)


def segment_eeg_into_epochs(eeg: np.ndarray, sfreq: float, epoch_len_s: float) -> Dict[str, object]:
    epoch_len_samples = int(epoch_len_s * sfreq)
    n_channels, n_samples = eeg.shape
    n_epochs = n_samples // epoch_len_samples
    trimmed = eeg[:, : n_epochs * epoch_len_samples]
    epochs = trimmed.reshape(n_channels, n_epochs, epoch_len_samples).transpose(1, 0, 2)

    start_samples = np.arange(n_epochs) * epoch_len_samples
    end_samples = start_samples + epoch_len_samples

    return {
        "epochs": epochs,
        "sfreq": sfreq,
        "epoch_start_times": start_samples / sfreq,
        "epoch_end_times": end_samples / sfreq,
        "n_epochs": n_epochs,
    }


def create_labels_from_intervals(seg: Dict[str, object], seizure_intervals: List[Tuple[float, float]]) -> np.ndarray:
    y = np.zeros(seg["n_epochs"], dtype=int)
    for i in range(seg["n_epochs"]):
        t0 = seg["epoch_start_times"][i]
        t1 = seg["epoch_end_times"][i]
        for s, e in seizure_intervals:
            if (t0 < e) and (t1 > s):
                y[i] = 1
                break
    return y


def extract_spatial_features_all_epochs(epochs: np.ndarray, fs: float, bands: Dict[str, Tuple[float, float]]) -> np.ndarray:
    n_epochs, n_channels, n_samples = epochs.shape
    eps = 1e-12
    freqs, psd = welch(epochs, fs=fs, nperseg=n_samples, axis=-1)
    feat_list = []

    for f_low, f_high in bands.values():
        idx = (freqs >= f_low) & (freqs < f_high)
        band_energy = simpson(psd[..., idx], freqs[idx], axis=-1) if np.any(idx) else np.zeros((n_epochs, n_channels))
        feat_list.append(band_energy)

    feat_list.append(np.sqrt(np.mean(epochs**2, axis=-1)))
    feat_list.append(np.sum(np.abs(np.diff(epochs, axis=-1)), axis=-1))

    std_x = np.std(epochs, axis=-1)
    dx = np.diff(epochs, axis=-1)
    std_dx = np.std(dx, axis=-1)
    ddx = np.diff(dx, axis=-1)
    std_ddx = np.std(ddx, axis=-1)
    feat_list.append((std_ddx / (std_dx + eps)) / ((std_dx / (std_x + eps)) + eps))

    feat_3d = np.stack(feat_list, axis=-1)
    return feat_3d.reshape(n_epochs, n_channels * feat_3d.shape[-1])


def temporal_stack_within_file(X_spatial: np.ndarray, y_epoch: np.ndarray, stack_window: int) -> Tuple[np.ndarray, np.ndarray]:
    n_epochs, n_feats = X_spatial.shape
    if n_epochs < stack_window:
        return np.empty((0, stack_window * n_feats)), np.empty((0,), dtype=int)
    X_final = np.vstack([X_spatial[i - (stack_window - 1): i + 1].reshape(-1) for i in range(stack_window - 1, n_epochs)])
    y_final = np.array([y_epoch[i] for i in range(stack_window - 1, n_epochs)], dtype=int)
    return X_final, y_final


## 4) Cache 层（重点）
> 第一次抽特征写入 cache；之后命中直接读取。

In [ ]:
def _cache_key(patient_dir: str, summary_path: str, channels: Sequence[str], epoch_len_s: float, stack_window: int, bandpass: Tuple[float, float], bands: Dict[str, Tuple[float, float]]) -> str:
    payload = {
        "patient_dir": os.path.abspath(patient_dir),
        "summary_path": os.path.abspath(summary_path),
        "channels": list(channels),
        "epoch_len_s": epoch_len_s,
        "stack_window": stack_window,
        "bandpass": bandpass,
        "bands": bands,
    }
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def _build_patient_features(patient_dir: str, summary_path: str, channels: Sequence[str], epoch_len_s: float, stack_window: int, bands: Dict[str, Tuple[float, float]], main_bandpass: Tuple[float, float]):
    seizure_dict = parse_summary_to_dict(summary_path)
    X_list, y_list, group_list, meta_list = [], [], [], []

    for path in list_edf_files(patient_dir):
        file_name = os.path.basename(path)
        seizure_intervals = seizure_dict.get(file_name, [])

        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        normalize_channel_names(raw)
        if not all(ch in raw.ch_names for ch in channels):
            raw.close(); continue

        raw.pick(list(channels))
        data = raw.get_data()
        fs = raw.info["sfreq"]

        data_bp = bandpass_filter_multich(data, fs, main_bandpass[0], main_bandpass[1])
        seg = segment_eeg_into_epochs(data_bp, fs, epoch_len_s)
        y_epoch = create_labels_from_intervals(seg, seizure_intervals)
        X_spatial = extract_spatial_features_all_epochs(seg["epochs"], fs, bands)
        X_final, y_final = temporal_stack_within_file(X_spatial, y_epoch, stack_window)

        if len(y_final) == 0:
            raw.close(); continue

        X_list.append(X_final)
        y_list.append(y_final)
        group_list.extend([file_name] * len(y_final))

        offset = stack_window - 1
        for idx in range(offset, seg["n_epochs"]):
            meta_list.append(EpochMetadata(file_name, float(seg["epoch_start_times"][idx]), float(seg["epoch_end_times"][idx])))

        raw.close()

    if not X_list:
        raise ValueError(f"No usable data for patient directory: {patient_dir}")

    return np.vstack(X_list), np.concatenate(y_list), np.array(group_list), meta_list


def load_or_create_patient_cache(patient_dir: str, summary_path: str, channels: Sequence[str], cache_dir: str, epoch_len_s: float, stack_window: int, bands: Dict[str, Tuple[float, float]], main_bandpass: Tuple[float, float]):
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    key = _cache_key(patient_dir, summary_path, channels, epoch_len_s, stack_window, main_bandpass, bands)
    cache_path = os.path.join(cache_dir, f"{Path(patient_dir).name}_{key}.npz")

    if os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        meta = [EpochMetadata(**m) for m in data["meta"]]
        return {"X": data["X"], "y": data["y"], "groups": data["groups"], "meta": meta, "cache_path": cache_path, "cache_hit": True}

    X, y, groups, meta = _build_patient_features(patient_dir, summary_path, channels, epoch_len_s, stack_window, bands, main_bandpass)
    np.savez_compressed(cache_path, X=X, y=y, groups=groups, meta=np.array([asdict(m) for m in meta], dtype=object))
    return {"X": X, "y": y, "groups": groups, "meta": meta, "cache_path": cache_path, "cache_hit": False}


## 5) 三模型投票 + 阈值选择
> 模型：SVM + RF + LR，先做 SelectKBest 再投票。

In [ ]:
def select_threshold(y_true: np.ndarray, y_score: np.ndarray, thresholds: np.ndarray) -> float:
    best_threshold = thresholds[0]
    best_balanced = -1.0
    for th in thresholds:
        y_pred = (y_score >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn + 1e-12)
        tnr = tn / (tn + fp + 1e-12)
        balanced = 0.5 * (tpr + tnr)
        if balanced > best_balanced:
            best_balanced = balanced
            best_threshold = th
    return float(best_threshold)


def build_models(k_features: int, random_state: int = 42) -> Dict[str, Pipeline]:
    return {
        "svm": Pipeline([("scaler", StandardScaler()), ("selector", SelectKBest(f_classif, k=k_features)), ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=random_state))]),
        "rf": Pipeline([("selector", SelectKBest(f_classif, k=k_features)), ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=random_state, n_jobs=-1))]),
        "lr": Pipeline([("scaler", StandardScaler()), ("selector", SelectKBest(f_classif, k=k_features)), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state))]),
    }


def fit_and_vote(models: Dict[str, Pipeline], X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray):
    scores = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        scores[name] = model.predict_proba(X_test)[:, 1]
    ensemble = np.mean(np.column_stack(list(scores.values())), axis=1)
    return scores, ensemble


## 6) 临床可视化
> 输出风险时间线 + 预测发作区间的多导联波形图。

In [ ]:
def epochs_to_intervals(times: Sequence[EpochMetadata], y_pred: np.ndarray):
    intervals = {}
    current_file = None
    current_start = None

    for meta, pred in zip(times, y_pred, strict=False):
        if current_file is None:
            current_file = meta.file_name

        if meta.file_name != current_file:
            if current_start is not None:
                intervals.setdefault(current_file, []).append((current_start, prev_end))
            current_file = meta.file_name
            current_start = None

        if pred == 1 and current_start is None:
            current_start = meta.start_s
        if pred == 0 and current_start is not None:
            intervals.setdefault(meta.file_name, []).append((current_start, prev_end))
            current_start = None

        prev_end = meta.end_s

    if current_start is not None and current_file is not None:
        intervals.setdefault(current_file, []).append((current_start, prev_end))

    return intervals


def plot_score_timeline(file_name: str, times: Sequence[EpochMetadata], scores: np.ndarray, threshold: float, output_dir: str):
    mask = np.array([m.file_name == file_name for m in times])
    if not np.any(mask):
        return

    plt.figure(figsize=(12, 3))
    plt.plot([m.start_s for m, f in zip(times, mask, strict=False) if f], scores[mask], label="Seizure probability")
    plt.axhline(threshold, color="red", linestyle="--", label=f"Threshold={threshold:.2f}")
    plt.xlabel("Time (s)")
    plt.ylabel("Probability")
    plt.title(f"Clinical timeline - {file_name}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{file_name}_score_timeline.png"), dpi=150)
    plt.close()


def plot_multichannel_waveform(raw: mne.io.BaseRaw, channels: Sequence[str], start_s: float, end_s: float, output_path: str):
    raw_crop = raw.copy().pick(list(channels)).crop(tmin=start_s, tmax=end_s)
    data = raw_crop.get_data()
    times = raw_crop.times
    offsets = np.arange(data.shape[0]) * np.nanmax(np.abs(data)) * 2.5

    plt.figure(figsize=(12, 6))
    for idx, ch in enumerate(channels):
        plt.plot(times, data[idx] + offsets[idx], linewidth=0.7, label=ch)
    plt.yticks([])
    plt.xlabel("Time (s)")
    plt.title(f"Raw EEG waveforms ({start_s:.1f}-{end_s:.1f}s)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


## 7) 主流程：默认 3 病人 LOOCV
> 你只要改 path 即可跑通整个流程。

In [ ]:
def run_unified_pipeline(patient_dirs: Sequence[str], summary_paths: Sequence[str], output_dir: str = "loo_outputs", cache_dir: str = "feature_cache", epoch_len_s: float = 2.0, stack_window: int = 3, bands: Dict[str, Tuple[float, float]] = DEFAULT_BANDS, main_bandpass: Tuple[float, float] = (0.5, 30.0), max_patients: int = 3, k_features: int = 256):
    if len(patient_dirs) != len(summary_paths):
        raise ValueError("Number of patient directories must match number of summary paths.")

    selected_pairs = list(zip(patient_dirs, summary_paths, strict=True))[:max_patients]
    patient_dirs = [p for p, _ in selected_pairs]
    summary_paths = [s for _, s in selected_pairs]

    os.makedirs(output_dir, exist_ok=True)
    channels = find_common_channels(patient_dirs)

    patient_data = {}
    for pdir, spath in zip(patient_dirs, summary_paths, strict=True):
        cached = load_or_create_patient_cache(pdir, spath, channels, cache_dir, epoch_len_s, stack_window, bands, main_bandpass)
        patient_data[pdir] = cached
        print(f"[CACHE] {Path(pdir).name}: {'HIT' if cached['cache_hit'] else 'MISS'} -> {cached['cache_path']}")

    for test_patient in patient_dirs:
        train_patients = [p for p in patient_dirs if p != test_patient]
        X_train = np.vstack([patient_data[p]["X"] for p in train_patients])
        y_train = np.concatenate([patient_data[p]["y"] for p in train_patients])
        groups_train = np.concatenate([patient_data[p]["groups"] for p in train_patients])

        X_test = patient_data[test_patient]["X"]
        y_test = patient_data[test_patient]["y"]
        meta_test = patient_data[test_patient]["meta"]

        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, val_idx = next(gss.split(X_train, y_train, groups_train))

        models = build_models(k_features=min(k_features, X_train.shape[1]))

        _, val_ensemble = fit_and_vote(models, X_train[train_idx], y_train[train_idx], X_train[val_idx])
        threshold = select_threshold(y_train[val_idx], val_ensemble, np.linspace(0.05, 0.95, 19))

        per_model_score, test_ensemble = fit_and_vote(models, X_train, y_train, X_test)
        y_pred = (test_ensemble >= threshold).astype(int)

        patient_name = os.path.basename(test_patient.rstrip("/"))
        patient_out_dir = os.path.join(output_dir, patient_name)
        os.makedirs(patient_out_dir, exist_ok=True)

        report_payload = {
            "patient": patient_name,
            "channels": channels,
            "threshold": threshold,
            "average_precision": average_precision_score(y_test, test_ensemble),
            "roc_auc": roc_auc_score(y_test, test_ensemble) if len(np.unique(y_test)) > 1 else float("nan"),
            "classification_report": classification_report(y_test, y_pred, output_dict=True, zero_division=0),
            "per_model_ap": {name: average_precision_score(y_test, s) for name, s in per_model_score.items()},
            "predicted_intervals": epochs_to_intervals(meta_test, y_pred),
        }

        with open(os.path.join(patient_out_dir, "summary.json"), "w", encoding="utf-8") as f:
            json.dump(report_payload, f, indent=2, ensure_ascii=False)

        for file_name in sorted(set(m.file_name for m in meta_test)):
            plot_score_timeline(file_name, meta_test, test_ensemble, threshold, patient_out_dir)

        for file_name, interval_list in report_payload["predicted_intervals"].items():
            edf_path = os.path.join(test_patient, file_name)
            if not os.path.exists(edf_path):
                continue
            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            normalize_channel_names(raw)
            for idx, (start_s, end_s) in enumerate(interval_list):
                output_path = os.path.join(patient_out_dir, f"{file_name}_interval_{idx + 1}_{start_s:.0f}-{end_s:.0f}.png")
                plot_multichannel_waveform(raw, channels, start_s, end_s, output_path)
            raw.close()


## 8) 运行示例
把下面路径替换为你自己的 3 个病人目录和 summary 路径，然后执行。

In [ ]:
# Example usage (edit to your local paths):
# patient_dirs = ["/path/to/chb01", "/path/to/chb02", "/path/to/chb03"]
# summary_paths = ["/path/to/chb01-summary.txt", "/path/to/chb02-summary.txt", "/path/to/chb03-summary.txt"]
#
# run_unified_pipeline(
#     patient_dirs=patient_dirs,
#     summary_paths=summary_paths,
#     output_dir="loo_outputs",
#     cache_dir="feature_cache",
#     max_patients=3,  # 快速跑通
# )
